# Karachi Crime Risk — Updated Preprocessing (Leakage-Safe + Memory-Safe)

This notebook fixes **data leakage** that can cause unrealistic perfect scores.

## Target
`HighRisk = 1` if `RISK_ZONE ∈ {Red, Orange}` else `0`

## Key fixes
- **Hard-drop leakage columns** (risk/priority/severity/labels)
- **Sparse one-hot encoding** to avoid memory blowups
- **Cap high-cardinality categories** using `min_frequency`
- Save features as **sparse `.npz`**

## Input (root / same folder)
- `karachi_crime_dataset.csv`

## Outputs (saved in same folder)
- `X_train.npz`, `X_test.npz`
- `y_train.csv`, `y_test.csv`
- `preprocessing_artifacts.joblib`


In [1]:
# 0) Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans

from scipy.sparse import save_npz
import joblib

RANDOM_STATE = 42


In [3]:
# 1) Load data (root path)
df = pd.read_csv("Raw Data/karachi_crime_dataset.csv")
print("Loaded:", df.shape)
df.head()


Loaded: (100000, 21)


,INCIDENT_ID,TOWN,TOWN_RISK_LEVEL,TOWN_PRIORITY_RANK,SUBDIVISION,SUBDIVISION_RISK_LEVEL,SUBDIVISION_PRIORITY_RANK,SEVERITY_SCORE,SEVERITY,DATE,...,LONGITUDE,CRIME_TYPE,IS_RED_ZONE,IS_ORANGE_ZONE,IS_YELLOW_ZONE,IS_GREEN_ZONE,IS_WHITE_ZONE,RISK_ZONE,SOURCE,RANK
0,INCIDENT_000001,Lyari Town,High,1,Baghdadi,High,1,10,High,2024-12-15,...,66.9998,Gang Violence,1,0,0,0,0,Red,Synthetic,1
1,INCIDENT_000002,Lyari Town,High,1,Baghdadi,High,1,10,High,2020-12-03,...,66.9998,Murder,1,0,0,0,0,Red,Synthetic,1
2,INCIDENT_000003,Lyari Town,High,1,Baghdadi,High,1,10,High,2025-01-24,...,66.9998,Murder,1,0,0,0,0,Red,Synthetic,1
3,INCIDENT_000004,Lyari Town,High,1,Baghdadi,High,1,10,High,2023-02-18,...,66.9998,Gang Violence,1,0,0,0,0,Red,Synthetic,1
4,INCIDENT_000005,Lyari Town,High,1,Baghdadi,High,1,10,High,2025-04-13,...,66.9998,Gang Violence,1,0,0,0,0,Red,Synthetic,1


## 2) Define target + HARD leakage block

In [4]:
# 2) Define target from RISK_ZONE
if "RISK_ZONE" not in df.columns:
    raise ValueError("RISK_ZONE column not found. It is required to build HighRisk target.")

df = df.copy()
df["HighRisk"] = df["RISK_ZONE"].isin(["Red", "Orange"]).astype(int)

print("Target distribution (HighRisk):")
print(df["HighRisk"].value_counts())
print(df["HighRisk"].value_counts(normalize=True).round(4))

# 🚫 HARD LEAKAGE BLOCK: remove ANY column that would trivially reveal HighRisk
LEAKAGE_COLS = [
    # Direct target or proxies
    "RISK_ZONE",
    "RISK_SCORE",
    "RISK_LEVEL",

    # Severity / priority derived from outcomes
    "SEVERITY_SCORE",
    "PRIORITY",
    "PRIORITY_RANK",
    "TOWN_PRIORITY_RANK",
    "SUBDIVISION_PRIORITY_RANK",

    # Labels / categories (if present)
    "CRIME_TYPE",
    "CRIME_CATEGORY",

    # Zone flags (if present)
    "IS_RED_ZONE", "IS_ORANGE_ZONE", "IS_YELLOW_ZONE", "IS_GREEN_ZONE", "IS_WHITE_ZONE",
    "is_red_zone", "is_orange_zone", "is_yellow_zone", "is_green_zone", "is_white_zone",
]

# Also drop any column name that contains 'risk' (except the target itself)
for c in df.columns:
    cl = c.lower()
    if ("risk" in cl) and (c not in ["HighRisk"]):
        LEAKAGE_COLS.append(c)

LEAKAGE_COLS = sorted(set(LEAKAGE_COLS))

print("\nLeakage columns to drop (if present):")
for c in LEAKAGE_COLS:
    if c in df.columns:
        print(" -", c)

# Build X/y
X = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns] + ["HighRisk"], errors="ignore")
y = df["HighRisk"].astype(int).values

print("\nRemaining feature columns:", len(X.columns))
print("Any leakage columns left?", any(c in X.columns for c in LEAKAGE_COLS))


Target distribution (HighRisk):
HighRisk
0    53723
1    46277
Name: count, dtype: int64
HighRisk
0    0.5372
1    0.4628
Name: proportion, dtype: float64

Leakage columns to drop (if present):
 - CRIME_TYPE
 - IS_GREEN_ZONE
 - IS_ORANGE_ZONE
 - IS_RED_ZONE
 - IS_WHITE_ZONE
 - IS_YELLOW_ZONE
 - RISK_ZONE
 - SEVERITY_SCORE
 - SUBDIVISION_PRIORITY_RANK
 - SUBDIVISION_RISK_LEVEL
 - TOWN_PRIORITY_RANK
 - TOWN_RISK_LEVEL

Remaining feature columns: 9
Any leakage columns left? False


## 3) Drop ID-like columns (near-unique strings)

In [5]:
# 3) Drop ID-like object columns with extremely high cardinality
id_like = []
n = len(X)

for c in X.columns:
    if X[c].dtype == "object":
        nunq = X[c].nunique(dropna=True)
        # If more than 50% values are unique, it's likely an identifier
        if nunq > 0.5 * n:
            id_like.append(c)

if id_like:
    print("Dropping ID-like columns:", id_like)
    X = X.drop(columns=id_like)
else:
    print("No ID-like columns detected.")


Dropping ID-like columns: ['INCIDENT_ID']


## 4) Train/Test split (stratified)

In [6]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train_raw.shape)
print("Test shape :", X_test_raw.shape)
print("Train positive rate:", float(y_train.mean()))
print("Test positive rate :", float(y_test.mean()))


Train shape: (80000, 8)
Test shape : (20000, 8)
Train positive rate: 0.462775
Test positive rate : 0.46275


## 5) Geo clustering transformer (optional but useful)

In [7]:
# 5) Geo cluster adder (fit on train only)
LAT_COL = "LATITUDE"
LON_COL = "LONGITUDE"

if LAT_COL not in X_train_raw.columns or LON_COL not in X_train_raw.columns:
    raise ValueError("LATITUDE/LONGITUDE not found in features. Ensure these columns exist in the CSV.")

class GeoClusterAdder(BaseEstimator, TransformerMixin):
    def __init__(self, lat_col=LAT_COL, lon_col=LON_COL, n_clusters=40, random_state=RANDOM_STATE):
        self.lat_col = lat_col
        self.lon_col = lon_col
        self.n_clusters = n_clusters
        self.random_state = random_state

    def fit(self, X, y=None):
        coords = X[[self.lat_col, self.lon_col]].copy()
        coords[self.lat_col] = pd.to_numeric(coords[self.lat_col], errors="coerce")
        coords[self.lon_col] = pd.to_numeric(coords[self.lon_col], errors="coerce")
        coords = coords.fillna(coords.median(numeric_only=True))
        self.km_ = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init="auto")
        self.km_.fit(coords)
        return self

    def transform(self, X):
        X_out = X.copy()
        coords = X_out[[self.lat_col, self.lon_col]].copy()
        coords[self.lat_col] = pd.to_numeric(coords[self.lat_col], errors="coerce")
        coords[self.lon_col] = pd.to_numeric(coords[self.lon_col], errors="coerce")
        coords = coords.fillna(coords.median(numeric_only=True))
        X_out["geo_cluster"] = self.km_.predict(coords).astype(np.int16)
        return X_out


## 6) Build preprocessing pipeline (sparse + controlled cardinality)

In [8]:
# 6) Identify numeric/categorical columns (geo_cluster will be added later)
num_cols = X_train_raw.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
cat_cols = X_train_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric cols:", len(num_cols))
print("Categorical cols:", len(cat_cols))

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))  # safe when combining with sparse
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True,
        min_frequency=200,     # ✅ controls feature explosion
        dtype=np.float32
    ))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    sparse_threshold=0.3,
    remainder="drop",
    verbose_feature_names_out=False
)

geo_adder = GeoClusterAdder()

full_pipeline = Pipeline([
    ("geo", geo_adder),
    ("prep", preprocess),
])


Numeric cols: 3
Categorical cols: 5


## 7) Fit/Transform + Save outputs

In [9]:
# 7) Fit/Transform
X_train = full_pipeline.fit_transform(X_train_raw, y_train)
X_test  = full_pipeline.transform(X_test_raw)

print("X_train:", X_train.shape, "nnz:", X_train.nnz)
print("X_test :", X_test.shape, "nnz:", X_test.nnz)

# Save sparse matrices
save_npz("X_train.npz", X_train)
save_npz("X_test.npz", X_test)

# Save targets
pd.DataFrame({"HighRisk": y_train}).to_csv("y_train.csv", index=False)
pd.DataFrame({"HighRisk": y_test}).to_csv("y_test.csv", index=False)

# Save artifacts (NOTE: contains GeoClusterAdder class)
artifacts = {
    "pipeline": full_pipeline,
    "feature_names": full_pipeline.named_steps["prep"].get_feature_names_out(),
    "lat_col": LAT_COL,
    "lon_col": LON_COL,
    "leakage_cols_dropped": [c for c in LEAKAGE_COLS if c in df.columns],
    "id_like_cols_dropped": id_like,
}

joblib.dump(artifacts, "preprocessing_artifacts.joblib")

print("\n✅ Saved: X_train.npz, X_test.npz, y_train.csv, y_test.csv, preprocessing_artifacts.joblib")


X_train: (80000, 158) nnz: 581331
X_test : (20000, 158) nnz: 145368

✅ Saved: X_train.npz, X_test.npz, y_train.csv, y_test.csv, preprocessing_artifacts.joblib


## 8) Sanity checks

In [10]:
# 8) Quick sanity checks
risk_like = [c for c in X.columns if "risk" in c.lower()]
print("Risk-like columns still in X:", risk_like)

feat_names = artifacts["feature_names"]
print("Feature count:", len(feat_names))
print("First 20 features:", feat_names[:20])


Risk-like columns still in X: []
Feature count: 158
First 20 features: ['LATITUDE' 'LONGITUDE' 'RANK' 'TOWN_Baldia Town' 'TOWN_Bin Qasim Town'
 'TOWN_Gadap Town' 'TOWN_Gulberg Town' 'TOWN_Gulshan-e-Iqbal Town'
 'TOWN_Jamshed Town' 'TOWN_Keamari Town' 'TOWN_Korangi Town'
 'TOWN_Landhi Town' 'TOWN_Liaquatabad Town' 'TOWN_Lyari Town'
 'TOWN_Malir Town' 'TOWN_New Karachi Town' 'TOWN_North Nazimabad Town'
 'TOWN_Orangi Town' 'TOWN_SITE Town' 'TOWN_Saddar Town']
